## Objective:
Build a system using the Ames Housing dataset to predict house prices and understand which factors affect the price most. The project also compares different regression models and checks where the predictions do not work well, especially for very expensive or very cheap houses.

## Project overview 
Housing prices depend on many things like location, size of the house, quality, and nearby area. Knowing the correct price is important for buyers, sellers, and real estate agents to make better decisions.

In this project, the goal is not only to predict house prices but also to understand which factors affect the price the most. The project also focuses on finding where the model works well and where it makes mistakes (where predictive models tend to fail ) so the results can be trusted in real-life situations.

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import numpy as np 

In [ ]:
dataset = pd.read_csv("AmesHousing.csv")

In [ ]:
dataset.columns

In [ ]:
dataset['Sale Condition'].unique()

In [ ]:
dataset[['Lot Area','Sale Condition','SalePrice']].sample(3)

## Problem Framing & Key Questions

The aim of this project is to predict house prices using past data and understand which factors affect prices the most. The focus is not only on prediction, but also on learning price patterns.

A successful model should give reasonable predictions on new data and clearly show which features influence the price. It should work well for most houses, not just the training data.

The main risk is wrong price prediction, which can mislead buyers or sellers. Overfitting or underfitting of model can cause financial loss, so model reliability is important.

The model may not perform well for very expensive houses, very old properties, or areas with limited data. These cases are harder to predict because there are fewer similar examples in the dataset.


In [ ]:
dataset.shape

In [ ]:
dataset['Sale Condition'].unique()

In [ ]:
dataset.info()

In [ ]:
dataset['Pool QC'].isnull().sum()

In [ ]:
dataset['MS Zoning'].unique()

In [ ]:
dataset.describe()

In [ ]:
dataset[['Lot Area','Lot Frontage','SalePrice']].corr()

#### AS We can see SalePrice is more dependent on Lot Frontage as compare to Lot Area as 12.7% of the variation in SalePrice can be explained by Lot Frontage & 7.1% of the variation in SalePrice can be explained by Lot Area.

In [ ]:
dataset[['Overall Qual','Gr Liv Area','SalePrice']].corr()

In [ ]:
dataset.select_dtypes(include=['int64','float64']).info()

In [ ]:
dataset.select_dtypes(include=['int64','float64']).columns

In [ ]:
dataset[['Order', 'PID', 'MS SubClass', 'Lot Frontage', 'Lot Area',
       'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add',
       'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF',
       'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd',
       'Fireplaces', 'Garage Yr Blt', 'Garage Cars', 'Garage Area',
       'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch',
       'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold',
       'SalePrice']].corr()

## As we see from corr matrix , several features of datatype(int64 , float64) show  strong linear relatioship with the target variable(Sale Price) : 
As all these features not contribute equally so I divide into category according to importance Tier1 , Tier2 , Tier3

## Tier 1: Strong Linear Association (Correlation > 0.70)

OverallQual (overall material and finish quality) , 
GrLivArea (above-ground living area)

## Tier 2: Moderate Linear Association (Correlation ≈ 0.50–0.70)

GarageCars, GarageArea , 
FullBath , 
TotalBsmtSF, 1stFlrSF , 
YearBuilt, YearRemodAdd
MasVnrArea , 
TotRmsAbvGrd 

Many of these features are interrelated, suggesting potential multicollinearity, particularly among size- and garage-related attributes.

## Tier 3: Weak to Moderate Linear Association (Correlation ≈ 0.20–0.50)

LotFrontage, LotArea ,
BsmtFinSF1, 2ndFlrSF ,
BsmtFullBath, HalfBath ,
Fireplaces , 
WoodDeckSF, OpenPorchSF

## Note : 
Correlation analysis reflects only linear relationships and does not imply causation. Some features with lower correlation may still play an important role through non-linear effects or interactions, which will be explored in subsequent analysis.

In [ ]:
dataset['Yr Sold'].unique()

### Here you can understand fron the visualization what is Linear relationship in correlation

In [ ]:
features = ['Lot Area', 'Gr Liv Area', 'Year Built','Overall Qual']

for feature in features:
    plt.figure(figsize=(12,6))

    # Scatter plot (Actual data)
    sns.scatterplot(
        x=feature,
        y='SalePrice',
        data=dataset,
        alpha=0.4,
        label='House Data'
    )

    # Linear regression line
    sns.regplot(
        x=feature,
        y='SalePrice',
        data=dataset,
        scatter=False,
        label='Linear Trend'
    )

    # Non-linear LOWESS trend
    sns.regplot(
        x=feature,
        y='SalePrice',
        data=dataset,
        scatter=False,
        lowess=True,
        line_kws={'linestyle': '--'},
        label='Actual Trend (LOWESS)'
    )

    plt.title(f'{feature} vs Sale Price')
    plt.xlabel(feature)
    plt.ylabel('Sale Price')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
dataset['Neighborhood'].unique()

In [ ]:
# 1. Create the sorting order
# We group by Neighborhood, find the median price for each, and sort them.
my_order = dataset.groupby(by=["Alley"])["SalePrice"].median().sort_values().index

# 2. Set the figure size (Needs to be wide for 25+ neighborhoods)
plt.figure(figsize=(14, 7))

# 3. Create the Boxplot with the 'order' parameter
sns.boxplot(x='Alley', y='SalePrice', data=dataset, order=my_order)

# 4. Rotate labels so they don't overlap
plt.xticks(rotation=45)
plt.title('Alley Ranked by Median Sale Price', fontsize=16)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
dataset['Overall Qual'].unique()

In [ ]:
dataset[['Overall Qual','Gr Liv Area','SalePrice']].corr()

### Question : Do we have very few expensive houses as compare to normal priced house ?

Answer : There are very few expensive houses compared to normal houses, so models don’t learn them well.

In [ ]:
dataset['SalePrice'].describe()

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(dataset['SalePrice'],bins=50,edgecolor='black',alpha=0.75)

plt.xlabel('Sale Price', fontsize=12)
plt.ylabel('Number of Houses', fontsize=12)
plt.title('Distribution of House Sale Prices', fontsize=14, fontweight='bold')

plt.show()


In [ ]:
dataset.select_dtypes(include='object').columns

In [ ]:
dataset[['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour',
       'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature',
       'Sale Type', 'Sale Condition']].head(1)

In [ ]:
dataset['Sale Condition'].unique()

## Some observations in categorical category 

In [ ]:
ordinal_cols = [ 'Lot shape' ,'Land Slope','Exter Qual','Exter Cond','Bsmt Qual','Bsmt Cond','Bsmt Exposure','BsmtFin Type 1','BsmtFin Type 2',
               'Heating QC','Electrical','Kitchen Qual','Functional','Fireplace Qu','Garage Finish','Garage Qual','Garage Cond',
                'Pool QC']

nominal_low = [ 'MS Zoning' ,'Street' , 'Alley', 'Land Contour','Utilities','Lot Config','Condition 1','Condition 2','Bldg Type','House Style','Roof Style'
              ,'Roof Matl','Mas Vnr Type','Foundation','Heating','Garage Type','Misc Feature','Sale Type','Sale Condition','Central Air','Paved Drive',
              'Fence']

nominal_high = [ 'Neighborhood','Exterior 1st','Exterior 2nd']


In [ ]:
# Handling Missing value / data 

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset.isnull().sum()[dataset.isnull().sum()>0]

In [ ]:
dataset.shape

In [ ]:
dataset['Pool QC'].unique()

## Identify type of Missing values 

In [ ]:
dataset.groupby('Neighborhood')['Lot Frontage'].median().sort_values()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
sns.boxplot(x='Neighborhood', y='Lot Frontage', data=dataset)
plt.xticks(rotation=45)
plt.show()


# Feature Engineering 

### step1 : Only Conceptual (not coding)

In [ ]:
dataset['House Age'] = dataset['Yr Sold'] - dataset['Year Built']   # House Age is better than Year Built because it shows the how much the house is old 